# Loanliness: data pipeline, balancing, clustering and visualisation

Reimplementation of the preprocessing / k-means / PCA + t-SNE parts of the paper. Sections follow the task list:
1. joined feature table, 2. cleaning, 3. encoding and scaling, 4. split, 5. balancing, 6. polynomial features, 7. k-means + per-cluster LightGBM, 8. PCA / t-SNE.

## 1. Build the joined feature table

In [1]:
import gc, json, warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

DATA = Path("dataset")
OUT = Path("artifacts")
OUT.mkdir(exist_ok=True)

CONFIG = {
    "seed": 42,
    "test_size": 0.2,
    "missing_col_thresh": 0.30,   # drop columns with more than this fraction missing
    "impute_mode": "mean",        # "mean": fill with train means (keep all rows), "drop_rows": paper-style
    "cat_label_max_card": 2,      # categoricals with <= this many levels are label encoded, rest one-hot
}

In [2]:
def agg_table(df, key, prefix, drop_cols=()):
    # numeric -> mean/min/max/sum, categorical -> one-hot then mean (share) and sum (count)
    df = df.drop(columns=list(drop_cols), errors="ignore")
    cat_cols = [c for c in df.select_dtypes(exclude="number").columns if c != key]
    num = df.drop(columns=cat_cols)
    num_cols = [c for c in num.columns if c != key]
    parts = [num.groupby(key)[num_cols].agg(["mean", "min", "max", "sum"])]
    if cat_cols:
        dummies = pd.get_dummies(df[cat_cols], dtype="float32", dummy_na=False)
        dummies[key] = df[key].values
        parts.append(dummies.groupby(key).agg(["mean", "sum"]))
    out = pd.concat(parts, axis=1)
    out.columns = [f"{prefix}_{a}_{b}".replace(" ", "_") for a, b in out.columns]
    out[f"{prefix}_COUNT"] = df.groupby(key).size().astype("float32")
    return out.astype("float32")


def read(name, **kw):
    df = pd.read_csv(DATA / name, **kw)
    for c in df.select_dtypes("float64").columns:
        df[c] = df[c].astype("float32")
    return df

In [3]:
# bureau_balance -> per bureau record -> merged into bureau -> per client
bb = read("bureau_balance.csv")
bb_agg = agg_table(bb, "SK_ID_BUREAU", "bb")
del bb; gc.collect()

bureau = read("bureau.csv")
bureau = bureau.merge(bb_agg, left_on="SK_ID_BUREAU", right_index=True, how="left")
bureau_agg = agg_table(bureau, "SK_ID_CURR", "bureau", drop_cols=["SK_ID_BUREAU"])
del bureau, bb_agg; gc.collect()
bureau_agg.shape

(305811, 179)

In [4]:
# previous_application: 365243 is a sentinel in the DAYS_* columns here too
prev = read("previous_application.csv")
day_cols = [c for c in prev.columns if c.startswith("DAYS_")]
prev[day_cols] = prev[day_cols].replace(365243, np.nan)
prev_agg = agg_table(prev, "SK_ID_CURR", "prev", drop_cols=["SK_ID_PREV"])
del prev; gc.collect()

pos = read("POS_CASH_balance.csv")
pos_agg = agg_table(pos, "SK_ID_CURR", "pos", drop_cols=["SK_ID_PREV"])
pos_agg["pos_N_LOANS"] = pos.groupby("SK_ID_CURR")["SK_ID_PREV"].nunique().astype("float32")
del pos; gc.collect()

inst = read("installments_payments.csv")
inst["DAYS_PAST_DUE"] = inst["DAYS_ENTRY_PAYMENT"] - inst["DAYS_INSTALMENT"]   # >0 means paid late
inst["PAYMENT_DIFF"] = inst["AMT_INSTALMENT"] - inst["AMT_PAYMENT"]            # >0 means underpaid
inst_agg = agg_table(inst, "SK_ID_CURR", "inst", drop_cols=["SK_ID_PREV"])
del inst; gc.collect()

cc = read("credit_card_balance.csv")
cc_agg = agg_table(cc, "SK_ID_CURR", "cc", drop_cols=["SK_ID_PREV"])
del cc; gc.collect()
[t.shape for t in (prev_agg, pos_agg, inst_agg, cc_agg)]

/tmp/ipykernel_46847/3161839987.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_COUNT"] = df.groupby(key).size().astype("float32")


[(338857, 363), (337252, 40), (339587, 33), (103558, 95)]

In [5]:
# left join everything onto application_train
app = read("application_train.csv")
app = app.rename(columns={c: f"app_{c}" for c in app.columns if c not in ("SK_ID_CURR", "TARGET")})
df = app
for agg in (bureau_agg, prev_agg, pos_agg, inst_agg, cc_agg):
    df = df.merge(agg, left_on="SK_ID_CURR", right_index=True, how="left")
del app, bureau_agg, prev_agg, pos_agg, inst_agg, cc_agg; gc.collect()

assert len(df) == 307511 and df["SK_ID_CURR"].is_unique
print(df.shape, "| default rate:", df["TARGET"].mean().round(4))
df.columns.to_series().str.split("_").str[0].value_counts()

(307511, 832) | default rate: 0.0807


prev      363
bureau    179
app       120
cc         95
pos        40
inst       33
SK          1
TARGET      1
Name: count, dtype: int64